# Silver Macro Transform

Heavy notebook for macro-domain Silver transforms.

Current implementation supports only the ECB branch:

- `brz_macro.raw_ecb_fx_ref_rates_daily`
- -> `slv_macro.ecb_fx_ref_rates_daily`
- rejected rows -> `slv_macro.ecb_fx_ref_rates_daily_quarantine`

Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import uuid
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

ECB_LOCAL_TZ = ZoneInfo("Europe/Berlin")


def parse_iso_date(field_name: str, raw_value: str):
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def parse_quote_currencies(raw_value: str) -> list[str]:
    quote_currencies = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    quote_currencies = list(dict.fromkeys(quote_currencies))
    if not quote_currencies:
        raise ValueError("quote_currencies cannot be empty")

    for currency in quote_currencies:
        if len(currency) != 3 or not currency.isalpha():
            raise ValueError(f"quote_currencies must contain 3-letter ISO codes, got: {currency}")

    return quote_currencies


def resolve_ecb_date_window(mode: str, start_date_raw: str, end_date_raw: str, lookback_days_raw: str):
    latest_complete_date = datetime.now(ECB_LOCAL_TZ).date() - timedelta(days=1)
    normalized_mode = mode.strip().lower()

    if normalized_mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if normalized_mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")
        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        try:
            lookback_days = int(lookback_days_raw or "0")
        except ValueError as exc:
            raise ValueError("lookback_days must be an integer in incremental mode") from exc

        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed Europe/Berlin day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def collect_counts(df, key_column: str) -> dict[str, int]:
    return {
        row[key_column]: row["row_count"]
        for row in df.groupBy(key_column).count().withColumnRenamed("count", "row_count").collect()
    }


dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("source_system", "ecb")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
source_system = dbutils.widgets.get("source_system").strip().lower()
quote_currencies = parse_quote_currencies(dbutils.widgets.get("quote_currencies").strip())
mode = dbutils.widgets.get("mode").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()
processed_at = datetime.now(timezone.utc)

if source_system != "ecb":
    raise ValueError("This heavy notebook currently supports only source_system='ecb'")

start_date, end_date = resolve_ecb_date_window(
    mode=mode,
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
)

source_table = f"{catalog}.brz_macro.raw_ecb_fx_ref_rates_daily"
target_table = f"{catalog}.slv_macro.ecb_fx_ref_rates_daily"
quarantine_table = f"{catalog}.slv_macro.ecb_fx_ref_rates_daily_quarantine"

for table_name in [source_table, target_table, quarantine_table]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

print(
    f"Transforming ECB Silver for {','.join(quote_currencies)} from {start_date.isoformat()} "
    f"to {end_date.isoformat()} in {mode} mode"
)

bronze_df = (
    spark.table(source_table)
    .select(
        F.upper(F.col("base_currency")).alias("base_currency"),
        F.upper(F.col("quote_currency")).alias("quote_currency"),
        F.col("rate_date"),
        F.col("rate"),
        F.col("ingested_at").alias("source_ingested_at"),
        F.col("run_id").alias("source_run_id"),
        F.col("payload_hash"),
    )
    .filter(F.col("quote_currency").isin(quote_currencies))
    .filter((F.col("rate_date") >= F.lit(start_date)) & (F.col("rate_date") <= F.lit(end_date)))
)

rows_read = bronze_df.count()
per_currency_rows_read = collect_counts(bronze_df, "quote_currency") if rows_read else {}

if rows_read == 0:
    result = {
        "status": "success_empty",
        "source_system": source_system,
        "mode": mode,
        "quote_currencies": quote_currencies,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "source_table": source_table,
        "target_table": target_table,
        "quarantine_table": quarantine_table,
        "rows_read": 0,
        "rows_after_dedup": 0,
        "rows_structural_invalid": 0,
        "rows_rejected": 0,
        "rows_quarantined": 0,
        "rows_to_update": 0,
        "rows_to_insert": 0,
        "rows_merged": 0,
        "run_id": run_id,
        "per_currency_rows_read": {},
        "per_currency_rows_after_dedup": {},
        "per_currency_rows_rejected": {},
        "per_currency_rows_merged": {},
    }
else:
    dedup_window = Window.partitionBy("base_currency", "quote_currency", "rate_date").orderBy(
        F.col("source_ingested_at").desc(),
        F.col("payload_hash").desc(),
    )

    deduped_df = (
        bronze_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rows_after_dedup = deduped_df.count()
    per_currency_rows_after_dedup = collect_counts(deduped_df, "quote_currency") if rows_after_dedup else {}

    structural_invalid_df = deduped_df.filter(
        F.col("base_currency").isNull()
        | F.col("quote_currency").isNull()
        | F.col("rate_date").isNull()
        | (~F.col("base_currency").rlike("^[A-Z]{3}$"))
        | (~F.col("quote_currency").rlike("^[A-Z]{3}$"))
        | (F.col("base_currency") != F.lit("EUR"))
    )
    rows_structural_invalid = structural_invalid_df.count()

    if rows_structural_invalid:
        display(structural_invalid_df.orderBy("quote_currency", "rate_date"))
        raise RuntimeError(
            f"Detected {rows_structural_invalid} structural-invalid ECB rows in Bronze. Silver load aborted."
        )

    rejected_df = deduped_df.filter(F.col("rate").isNull() | (F.col("rate") <= F.lit(0)))
    rows_rejected = rejected_df.count()
    per_currency_rows_rejected = collect_counts(rejected_df, "quote_currency") if rows_rejected else {}

    if rows_rejected:
        quarantined_df = rejected_df.select(
            F.lit(run_id).alias("run_id"),
            F.col("base_currency"),
            F.col("quote_currency"),
            F.col("rate_date"),
            F.col("rate"),
            F.lit("rate_missing_or_non_positive").alias("dq_reason"),
            F.col("source_ingested_at"),
            F.col("source_run_id"),
            F.col("payload_hash"),
            F.lit(processed_at).alias("quarantined_at"),
        )
        quarantined_df.write.format("delta").mode("append").saveAsTable(quarantine_table)
        display(quarantined_df.orderBy("quote_currency", "rate_date"))
    else:
        quarantined_df = None

    valid_df = deduped_df.filter(F.col("rate").isNotNull() & (F.col("rate") > F.lit(0))).select(
        F.col("base_currency"),
        F.col("quote_currency"),
        F.col("rate_date"),
        F.col("rate"),
        F.lit(processed_at).alias("ingested_at"),
        F.lit(run_id).alias("run_id"),
    )

    rows_merged = valid_df.count()
    per_currency_rows_merged = collect_counts(valid_df, "quote_currency") if rows_merged else {}
    existing_key_count = (
        valid_df.select("base_currency", "quote_currency", "rate_date")
        .join(
            spark.table(target_table).select("base_currency", "quote_currency", "rate_date"),
            on=["base_currency", "quote_currency", "rate_date"],
            how="inner",
        )
        .count()
    )

    DeltaTable.forName(spark, target_table).alias("tgt").merge(
        valid_df.alias("src"),
        "tgt.base_currency = src.base_currency AND tgt.quote_currency = src.quote_currency AND tgt.rate_date = src.rate_date",
    ).whenMatchedUpdate(
        set={
            "rate": "src.rate",
            "ingested_at": "src.ingested_at",
            "run_id": "src.run_id",
        }
    ).whenNotMatchedInsertAll().execute()

    display(valid_df.orderBy("quote_currency", "rate_date"))

    result = {
        "status": "success",
        "source_system": source_system,
        "mode": mode,
        "quote_currencies": quote_currencies,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "source_table": source_table,
        "target_table": target_table,
        "quarantine_table": quarantine_table,
        "rows_read": rows_read,
        "rows_after_dedup": rows_after_dedup,
        "rows_structural_invalid": 0,
        "rows_rejected": rows_rejected,
        "rows_quarantined": rows_rejected,
        "rows_to_update": existing_key_count,
        "rows_to_insert": rows_merged - existing_key_count,
        "rows_merged": rows_merged,
        "run_id": run_id,
        "per_currency_rows_read": per_currency_rows_read,
        "per_currency_rows_after_dedup": per_currency_rows_after_dedup,
        "per_currency_rows_rejected": per_currency_rows_rejected,
        "per_currency_rows_merged": per_currency_rows_merged,
    }

result
